Importer les libraieries et les évenements

In [1]:
import pandas as pd

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

df = pd.read_csv("data/processed/openagenda_clean.csv")

print("Nombre d'événements :", len(df))

Nombre d'événements : 11718


Transformer les documents

In [2]:
documents = []

for _, row in df.iterrows():
    doc = Document(
        page_content=str(row["texte_rag"]),
        metadata={
            "id": str(row["Identifiant"]),
            "titre": str(row["Titre"]),
            "ville": str(row["Ville"]),
            "lieu": str(row["Nom du lieu"]),
            "date_debut": str(row["Première date - Début"]),
            "date_fin": str(row["Dernière date - Fin"]),
            "url": str(row["URL canonique"])
        }
    )

    documents.append(doc)

print("Documents créés :", len(documents))

Documents créés : 11718


Deecouper les documents en chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, #taille de frament à 500 caractères
    chunk_overlap=100 #crée le chevauchement entre les chunks pour éviter de perdre du contexte lors de la génération des embeddings
)

chunks = text_splitter.split_documents(documents)

print("Nombre de documents :", len(documents))
print("Nombre de chunks :", len(chunks))

Nombre de documents : 11718
Nombre de chunks : 38171


In [4]:
print(chunks[0].page_content)
print(chunks[0].metadata)

Événement : Dimanche au Vert en famille avec enfants de 3 à 10 ans. Paris 19.. Description : Balade Nature & Montessori. (Re)connectez-vous à la Nature en Famille. En DUO 1 enfant/ 1 adulte. Informations : Les Dimanches au Vert en famille proposent aux parents et leurs enfants de 3 à 10 ans des expériences Nature pour leur permettre de vivre ensemble des aventures riches et complices au cœur de 2 lieux parisiens ou proche de Paris les dimanches de 10 h à 12h. Chaque Saison à travers nos 5 sens.
{'id': '97781943', 'titre': 'Dimanche au Vert en famille avec enfants de 3 à 10 ans. Paris 19.', 'ville': 'Paris', 'lieu': 'Parc des Buttes-Chaumont, 75019 Paris, France', 'date_debut': '2024-01-28 10:00:00+00:00', 'date_fin': '2026-06-07 12:00:00+00:00', 'url': 'https://openagenda.com/econature/events/decouverte-nature-en-famille-avec-enfants-de-2-a-10-ans-paris-19'}


In [5]:
ids_chunks = {chunk.metadata["id"] for chunk in chunks}

print("Événements initiaux :", df["Identifiant"].nunique())
print("Événements présents dans les chunks :", len(ids_chunks))
print("Nombre total de chunks :", len(chunks))

Événements initiaux : 11718
Événements présents dans les chunks : 11718
Nombre total de chunks : 38171


Vectorisation des chunks

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"batch_size": 32}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Création de l'index avec Langchain

In [8]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=model
)

vectorstore.save_local("data/faiss_index")

In [9]:
print("Nombre de chunks :", len(chunks))
print("Vecteurs indexés :", vectorstore.index.ntotal)
print("Dimensions :", vectorstore.index.d)

Nombre de chunks : 38171
Vecteurs indexés : 38171
Dimensions : 384


Test de la première recherche semantique

In [10]:
question = "Quelles activités nature faire avec des enfants à Paris ?"

resultats = vectorstore.similarity_search(
    question,
    k=5
)

for doc in resultats:
    print("Titre :", doc.metadata["titre"])
    print("Lieu :", doc.metadata["lieu"])
    print("Date de fin :", doc.metadata["date_fin"])
    print("Description :", doc.page_content[:200])
    print("-" * 50)

Titre : FORUM DE L'EMPLOI MAIRIE DU 20e - La Ville de Paris recrute des Assistants Educatifs Petite Enfance (F/H) en CDI
Lieu : Agence Paris 20eme Vitruve
Date de fin : 2026-06-08 15:00:00+00:00
Description : Enfance, vous exercez au sein de l'une des 406 crèches de la Ville de Paris et vous participez, aux côtés des auxiliaires de puériculture, à l'accueil et l'accompagnement des enfants. vous serez amené
--------------------------------------------------
Titre : Jeux et activités pour enfants autour de l'histoire de Paris
Lieu : Archives de Paris
Date de fin : 2026-09-20 16:00:00+00:00
Description : Événement : Jeux et activités pour enfants autour de l'histoire de Paris. Description : Jeux pour enfants. Informations : Les Archives de Paris proposent aux enfants de découvrir l'histoire de la capi
--------------------------------------------------
Titre : Forum des métiers Travailler en crèche municipale
Lieu : Quartier Jeunes (Ancienne Mairie du 1er)
Date de fin : 2026-04-16 11:30:00

Afficher les distances de similarités

In [11]:
resultats = vectorstore.similarity_search_with_score(
    "Quels concerts de jazz sont organisés à Paris ?",
    k=5
)

for doc, score in resultats:
    print("Titre :", doc.metadata["titre"])
    print("Distance :", round(float(score), 4))
    print("-" * 40)

Titre : MARATHON JAZZ
Distance : 12.2533
----------------------------------------
Titre : Jazzycolors - Festival de jazz international
Distance : 12.4517
----------------------------------------
Titre : Jazzycolors - Festival de jazz international
Distance : 14.2923
----------------------------------------
Titre : Le ministère de la Culture célèbre le jazz dans les jardins du Palais-Royal
Distance : 14.3822
----------------------------------------
Titre : Jazz à la Cité
Distance : 14.5088
----------------------------------------


tester le rechargement pour que l'API utilise la base FAISS sans recontruire l'ensemble des embeddings

In [12]:
from langchain_community.vectorstores import FAISS

index_charge = FAISS.load_local(
    "data/faiss_index",
    embeddings=model,
    allow_dangerous_deserialization=True
)

print("Vecteurs rechargés :", index_charge.index.ntotal)

Vecteurs rechargés : 38171


Preparer retriever pour le systeme Rag

In [13]:
retriever = index_charge.as_retriever(
    search_kwargs={"k": 5}
)

documents_retrouves = retriever.invoke(
    "Je cherche une exposition à Paris"
)

for doc in documents_retrouves:
    print(doc.metadata["titre"])

Salon Fine Arts Paris
Salon Fine Arts Paris
Visite de l'Hôtel de Ville de Paris (sur inscription)
Des photographies aux archives : pour quoi faire ?
Visite libre du musée des égouts de Paris


S'assurer que l'index est optimisé pour des recherches rapides

In [14]:
import time

question = "Quels concerts sont prévus à Paris ?"

debut = time.perf_counter()

resultats = vectorstore.similarity_search(question, k=5)

temps = time.perf_counter() - debut

print(f"Temps de recherche : {temps:.3f} secondes")
print(f"Nombre de résultats : {len(resultats)}")

Temps de recherche : 0.358 secondes
Nombre de résultats : 5
